In [ ]:
import torch
import editdistance

from unsloth import FastVisionModel
from transformers import AutoProcessor

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

from datasets import load_dataset
from common import convert_row

In [ ]:
MAX_SEQ_LENGTH = 20000
TARGET_FPS = 24
RESOLUTION = (512, 360)

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained("outputs/checkpoint-400")
FastVisionModel.for_inference(model)

In [ ]:
dataset = load_dataset("Flimdejong/how2sign-3s", split="test")

In [ ]:
def predict(row):
	messages, frames = convert_row(row, target_fps=TARGET_FPS, resolution=RESOLUTION, include_description=False)

	text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
	inputs = tokenizer(text=text, videos=[frames], return_tensors="pt").to("cuda")

	with torch.no_grad():
		output_ids = model.generate(
			**inputs,
			max_new_tokens=128,
			use_cache=True,
			do_sample=False,  # greedy for evaluation
			repetition_penalty=1.0,
			eos_token_id=tokenizer.eos_token_id,
		)

	input_len = inputs["input_ids"].shape[1]
	generated = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
	return generated


def wer(ref, hyp):
	ref_words = ref.lower().split()
	hyp_words = hyp.lower().split()
	distance = editdistance.eval(ref_words, hyp_words)
	return distance / max(len(ref_words), 1)


def cer(ref, hyp):
	distance = editdistance.eval(ref.lower(), hyp.lower())
	return distance / max(len(ref), 1)

In [ ]:
# --- Eval loop ---
n = 100
results = []

for i in range(n):
	row = dataset[i]
	gt = row["sentence"].strip()

	try:
		pred = predict(row)
	except Exception as e:
		print(f"[{i}] ERROR: {e}")
		pred = ""

	w = wer(gt, pred)
	c = cer(gt, pred)
	exact = int(pred.lower() == gt.lower())
	results.append({"gt": gt, "pred": pred, "wer": w, "cer": c, "exact": exact})

	# if i % 10 == 0:
	print(f"[{i:03d}] WER={w:.2f} | GT: {gt!r} | PRED: {pred!r}")

In [ ]:
# --- Summary ---
avg_wer = sum(r["wer"] for r in results) / n
avg_cer = sum(r["cer"] for r in results) / n
exact_acc = sum(r["exact"] for r in results) / n

print("\n===== EVAL RESULTS (n=100) =====")
print(f"  Avg WER:        {avg_wer:.4f}  (lower is better, 0 = perfect)")
print(f"  Avg CER:        {avg_cer:.4f}  (lower is better, 0 = perfect)")
print(f"  Exact Match:    {exact_acc:.4f}  ({sum(r['exact'] for r in results)}/100 correct)")


# Show best predictions
print("\n--- Best 5 by WER ---")
for r in sorted(results, key=lambda x: x["wer"])[:5]:
	print(f"  WER={r['wer']:.2f} | GT: {r['gt']!r} | PRED: {r['pred']!r}")

# Show worst predictions
print("\n--- Worst 5 by WER ---")
for r in sorted(results, key=lambda x: -x["wer"])[:5]:
	print(f"  WER={r['wer']:.2f} | GT: {r['gt']!r} | PRED: {r['pred']!r}")